In [75]:
%load_ext autoreload
%autoreload 2

In [76]:
import numpy as np
import polars as pl

In [2]:
clinicalMatrix = pl.read_csv("xena_brca_data/TCGA.BRCA.sampleMap_BRCA_clinicalMatrix", separator="\t", infer_schema_length=0)

cM = clinicalMatrix.filter(pl.col("PAM50_mRNA_nature2012") != "null")
cMrnaseq= clinicalMatrix.filter(pl.col("PAM50Call_RNAseq") != "null")

In [3]:
cM["sampleID"][0]

'TCGA-A1-A0SD-01'

In [4]:
def get_counts_in_col(df, col_name):
    return np.unique(df[col_name].to_numpy(), return_counts=True)

get_counts_in_col(cM, "PAM50_mRNA_nature2012")

(array(['Basal-like', 'HER2-enriched', 'Luminal A', 'Luminal B',
        'Normal-like'], dtype=object),
 array([ 98,  58, 231, 127,   8]))

In [5]:
get_counts_in_col(cMrnaseq, "PAM50Call_RNAseq")

(array(['Basal', 'Her2', 'LumA', 'LumB', 'Normal'], dtype=object),
 array([142,  67, 434, 194, 119]))

In [13]:
rna = pl.read_csv("xena_brca_data/TCGA.BRCA.sampleMap_HiSeqV2", separator="\t")
cna = pl.read_csv("xena_brca_data/TCGA.BRCA.sampleMap_Gistic2_CopyNumber_Gistic2_all_thresholded.by_genes", separator="\t")
mirna = pl.read_csv("xena_brca_data/TCGA.BRCA.sampleMap_miRNA_HiSeq_gene", separator="\t", null_values=["NA"])

In [7]:
rna.head()

sample,TCGA-AR-A5QQ-01,TCGA-D8-A1JA-01,TCGA-BH-A0BQ-01,TCGA-BH-A0BT-01,TCGA-A8-A06X-01,TCGA-A8-A096-01,TCGA-BH-A0C7-01,TCGA-AC-A5XU-01,TCGA-PE-A5DE-01,TCGA-PE-A5DC-01,TCGA-AR-A0TV-01,TCGA-GM-A3XG-01,TCGA-BH-A18J-01,TCGA-BH-A0W7-01,TCGA-E9-A3QA-01,TCGA-A7-A4SD-01,TCGA-BH-A0HA-01,TCGA-AR-A5QN-01,TCGA-A7-A0CH-11,TCGA-A7-A0CE-01,TCGA-AR-A0U1-01,TCGA-EW-A1OZ-01,TCGA-A2-A0EY-01,TCGA-A8-A09R-01,TCGA-LL-A440-01,TCGA-BH-A8FY-01,TCGA-E2-A1II-01,TCGA-A7-A6VX-01,TCGA-C8-A273-01,TCGA-BH-A1EO-01,TCGA-OL-A5RX-01,TCGA-BH-A0B9-01,TCGA-EW-A1P5-01,TCGA-AO-A03P-01,TCGA-AN-A0AS-01,TCGA-A2-A1G0-01,…,TCGA-BH-A5IZ-01,TCGA-A8-A09W-01,TCGA-E2-A107-01,TCGA-AR-A252-01,TCGA-C8-A12Z-01,TCGA-BH-A202-01,TCGA-AO-A1KT-01,TCGA-D8-A1XW-01,TCGA-D8-A1JU-01,TCGA-E9-A1N5-01,TCGA-A2-A4S1-01,TCGA-E2-A1IG-11,TCGA-E2-A153-01,TCGA-A2-A0YG-01,TCGA-BH-A0B7-01,TCGA-D8-A1X6-01,TCGA-BH-A0BL-01,TCGA-BH-A0DO-01,TCGA-A2-A04Q-01,TCGA-BH-A0B5-11,TCGA-BH-A1FE-06,TCGA-E9-A1NI-01,TCGA-BH-A0HY-01,TCGA-AR-A24T-01,TCGA-E9-A1NF-11,TCGA-AO-A0JA-01,TCGA-D8-A1XZ-01,TCGA-A7-A13E-11,TCGA-C8-A8HP-01,TCGA-E9-A5FL-01,TCGA-AC-A2FB-11,TCGA-E2-A15F-01,TCGA-A2-A3XT-01,TCGA-B6-A0X7-01,TCGA-BH-A1EV-11,TCGA-3C-AALJ-01,TCGA-B6-A0X1-01
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ARHGEF10L""",9.5074,7.4346,9.3216,9.0198,9.6417,9.7665,10.0931,9.1524,9.9398,9.6287,9.6694,10.2287,9.2718,9.561,11.3338,11.3722,9.8078,10.1081,10.0819,9.5751,9.6685,9.0602,9.5451,9.6014,10.452,9.886,9.187,9.6146,9.7165,9.7915,10.1624,10.0971,9.7215,9.0184,8.7282,10.0161,…,11.6229,7.5288,10.3434,8.8947,8.8813,10.0359,8.6075,10.9642,9.7902,10.165,10.3455,10.0335,9.984,8.8155,9.1984,8.8809,9.5235,10.4995,10.064,9.3597,8.1078,9.7984,9.34,8.6545,10.4037,8.9976,8.5721,9.6265,10.1826,9.9199,9.909,10.0334,11.5144,10.5745,9.4048,10.9468,10.3164
"""HIF3A""",1.5787,3.6607,2.7224,1.3414,0.5819,0.2738,3.609,0.4738,2.9378,4.1136,0.433,5.342,2.1682,2.6648,2.5048,3.6372,1.9196,5.739,8.5517,4.1385,4.5824,0.7568,1.3008,0.656,6.4486,2.5831,5.4853,2.9753,2.1762,2.6206,3.7467,9.5132,0.4654,0.4157,0.0,6.2594,…,0.5021,0.8787,2.6529,4.4423,0.577,1.359,1.6344,2.3822,4.7678,2.154,2.1386,9.0708,1.6892,0.0,4.3177,0.5947,6.2779,3.9977,2.2799,8.8463,7.7224,4.227,0.0,0.4028,9.4111,0.7707,1.5668,8.1546,2.2159,3.8645,8.1872,0.8836,1.3169,4.0696,7.2537,0.931,2.4191
"""RNF17""",0.0,0.6245,0.5526,0.0,0.0,0.8765,0.0,0.0,0.0,0.0,0.0,0.0,0.4324,0.0,0.0,0.0,0.0,0.0,0.0,1.4303,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.9631,0.0,0.0,0.0,0.0,1.2109,…,0.0,0.0,0.0,0.3744,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.638,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.2703,0.0,0.0,0.0,3.7305,0.0,0.0,1.1329,0.4258,0.0,0.0,0.0
"""RNF10""",11.3676,11.9181,11.9665,13.1881,12.0036,11.8118,11.382,11.5004,12.2055,12.1312,11.9378,11.8971,12.3127,11.9861,11.1646,11.5724,12.4059,11.7531,11.9788,11.4626,12.4277,12.0018,11.5879,11.597,11.5529,12.0487,12.5989,11.3869,12.3921,11.5845,11.7273,12.0289,12.1964,12.2011,11.5278,11.7931,…,11.343,11.821,12.4371,11.6447,11.5965,11.4955,11.9798,11.4954,11.7845,11.8549,11.2387,11.8407,11.7244,12.2153,11.4783,12.1456,11.4204,11.8661,11.7065,12.1359,12.2541,12.5475,11.4961,11.6361,11.8686,12.0733,11.9794,11.9869,12.2653,12.4815,11.8263,12.0135,11.5818,11.8663,11.546,12.2616,12.157
"""RNF11""",11.1292,13.5273,11.4105,11.0911,11.2545,10.8554,10.7663,10.4358,11.221,10.8013,11.2889,11.3988,11.2739,10.841,10.8042,11.2059,11.4196,10.9013,11.5315,10.5215,10.5884,10.4395,11.1212,10.7365,10.9343,10.6782,10.8347,11.1685,10.7344,11.7243,11.236,9.6044,11.3045,11.2983,11.0501,11.8753,…,10.7503,11.4714,10.5106,11.4956,12.0406,11.2599,10.7881,11.6882,11.7708,11.1607,11.3168,11.7084,11.1408,11.3417,12.3537,11.1187,11.0695,11.1226,10.7574,12.3855,11.3686,10.6631,11.5667,11.1444,11.8553,11.1703,10.8409,11.9344,11.4117,10.4902,

In [8]:
cna.head()

Gene Symbol,TCGA-3C-AAAU-01,TCGA-3C-AALI-01,TCGA-3C-AALJ-01,TCGA-3C-AALK-01,TCGA-4H-AAAK-01,TCGA-5L-AAT0-01,TCGA-5L-AAT1-01,TCGA-5T-A9QA-01,TCGA-A1-A0SB-01,TCGA-A1-A0SD-01,TCGA-A1-A0SE-01,TCGA-A1-A0SF-01,TCGA-A1-A0SG-01,TCGA-A1-A0SH-01,TCGA-A1-A0SI-01,TCGA-A1-A0SJ-01,TCGA-A1-A0SK-01,TCGA-A1-A0SM-01,TCGA-A1-A0SN-01,TCGA-A1-A0SO-01,TCGA-A1-A0SP-01,TCGA-A1-A0SQ-01,TCGA-A2-A04N-01,TCGA-A2-A04P-01,TCGA-A2-A04Q-01,TCGA-A2-A04R-01,TCGA-A2-A04T-01,TCGA-A2-A04U-01,TCGA-A2-A04V-01,TCGA-A2-A04W-01,TCGA-A2-A04X-01,TCGA-A2-A04Y-01,TCGA-A2-A0CK-01,TCGA-A2-A0CL-01,TCGA-A2-A0CM-01,TCGA-A2-A0CO-01,…,TCGA-OL-A66J-01,TCGA-OL-A66K-01,TCGA-OL-A66L-01,TCGA-OL-A66N-01,TCGA-OL-A66O-01,TCGA-OL-A66P-01,TCGA-OL-A6VO-01,TCGA-OL-A6VQ-01,TCGA-OL-A6VR-01,TCGA-OL-A97C-01,TCGA-PE-A5DC-01,TCGA-PE-A5DD-01,TCGA-PE-A5DE-01,TCGA-PL-A8LV-01,TCGA-PL-A8LX-01,TCGA-PL-A8LY-01,TCGA-PL-A8LZ-01,TCGA-S3-A6ZF-01,TCGA-S3-A6ZG-01,TCGA-S3-A6ZH-01,TCGA-S3-AA0Z-01,TCGA-S3-AA10-01,TCGA-S3-AA11-01,TCGA-S3-AA12-01,TCGA-S3-AA14-01,TCGA-S3-AA15-01,TCGA-S3-AA17-01,TCGA-UL-AAZ6-01,TCGA-UU-A93S-01,TCGA-V7-A7HQ-01,TCGA-W8-A86G-01,TCGA-WT-AB41-01,TCGA-WT-AB44-01,TCGA-XX-A899-01,TCGA-XX-A89A-01,TCGA-Z7-A8R5-01,TCGA-Z7-A8R6-01
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""ACAP3""",0,-1,-1,0,0,0,0,-1,0,-1,0,0,0,-1,0,0,1,-1,-1,-1,1,0,0,1,0,-1,0,2,0,1,-1,1,0,0,1,0,…,-2,-1,0,-1,0,0,1,0,0,-2,-1,-1,0,1,-1,0,1,-1,-1,-1,0,0,0,0,0,0,-1,0,-1,0,0,-1,-1,0,-1,-1,0
"""ACTRT2""",0,-1,-1,0,0,0,0,-1,0,-1,0,0,0,-1,0,0,1,-1,-1,-1,1,0,0,1,0,-1,0,2,0,1,-1,1,0,0,1,0,…,-2,-1,0,-1,0,0,1,0,0,-2,-1,-1,0,1,-1,0,1,-1,-1,-1,0,0,0,0,0,0,-1,0,-1,0,0,-1,-1,0,-1,-1,0
"""AGRN""",0,-1,-1,0,0,0,0,-1,0,-1,0,0,0,-1,0,0,1,-1,-1,-1,1,0,0,1,0,-1,0,2,0,1,-1,1,0,0,1,0,…,-2,-1,0,-1,0,0,1,0,0,-2,-1,-1,0,1,-1,0,1,-1,-1,-1,0,0,0,0,0,0,-1,0,-1,0,0,-1,-1,0,-1,-1,0
"""ANKRD65""",0,-1,-1,0,0,0,0,-1,0,-1,0,0,0,-1,0,0,1,-1,-1,-1,1,0,0,1,0,-1,0,2,0,1,-1,1,0,0,1,0,…,-2,-1,0,-1,0,0,1,0,0,-2,-1,-1,0,1,-1,0,1,-1,-1,-1,0,0,0,0,0,0,-1,0,-1,0,0,-1,-1,0,-1,-1,0
"""ATAD3A""",0,-1,-1,0,0,0,0,-1,0,-1,0,0,0,-1,0,0,1,-1,-1,-1,1,0,0,1,0,-1,0,2,0,1,-1,1,0,0,1,0,…,-2,-1,0,-1,0,0,1,0,0,-2,-1,-1,0,1,-1,0,1,-1,-1,-1,0,0,0,0,0,0,-1,0,-1,0,0,-1,-1,0,-1,-1,0


In [9]:
# get common samples
rna_sample_names = rna.columns[1:]

In [14]:
mirna.head()

sample,TCGA-OL-A66H-01,TCGA-3C-AALK-01,TCGA-AR-A1AH-01,TCGA-AC-A5EH-01,TCGA-EW-A2FW-01,TCGA-E9-A1R0-01,TCGA-BH-A0BL-01,TCGA-AR-A1AJ-01,TCGA-A7-A13G-01,TCGA-AC-A62Y-01,TCGA-E9-A1QZ-01,TCGA-E9-A1R2-01,TCGA-GM-A2DK-01,TCGA-AR-A255-01,TCGA-E2-A15T-01,TCGA-AO-A1KQ-01,TCGA-E9-A1N4-11,TCGA-C8-A1HL-01,TCGA-D8-A1JD-01,TCGA-D8-A1XA-01,TCGA-E9-A5FL-01,TCGA-BH-A1F2-11,TCGA-A7-A3RF-01,TCGA-A7-A3IY-01,TCGA-GM-A2D9-01,TCGA-D8-A3Z5-01,TCGA-E9-A1ND-01,TCGA-EW-A1P5-01,TCGA-E9-A1RD-01,TCGA-AR-A250-01,TCGA-E2-A153-11,TCGA-AR-A1AQ-01,TCGA-LD-A66U-01,TCGA-BH-A1EN-01,TCGA-AO-A1KO-01,TCGA-BH-A0AU-11,…,TCGA-AC-A2FG-01,TCGA-AR-A1AX-01,TCGA-AR-A1AL-01,TCGA-BH-A1ET-01,TCGA-C8-A12W-01,TCGA-D8-A13Z-01,TCGA-BH-A0BZ-11,TCGA-D8-A27K-01,TCGA-BH-A28Q-01,TCGA-E9-A1NG-11,TCGA-EW-A1J1-01,TCGA-BH-A18K-11,TCGA-C8-A1HG-01,TCGA-D8-A1XK-01,TCGA-A7-A4SA-01,TCGA-3C-AAAU-01,TCGA-A2-A4S0-01,TCGA-BH-A0DV-11,TCGA-D8-A27E-01,TCGA-AC-A8OQ-01,TCGA-BH-A0BS-11,TCGA-EW-A1PB-01,TCGA-D8-A1JU-01,TCGA-E9-A1R7-11,TCGA-D8-A147-01,TCGA-BH-A0BT-01,TCGA-E9-A1N8-01,TCGA-E2-A574-01,TCGA-E2-A1IG-01,TCGA-5L-AAT0-01,TCGA-E9-A1NH-01,TCGA-EW-A6SA-01,TCGA-AR-A24Q-01,TCGA-BH-A1F0-01,TCGA-E9-A1RB-11,TCGA-GM-A4E0-01,TCGA-AC-A5XU-01
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""MIMAT0019868""",0.2381,null,null,null,0.1951,null,null,0.4831,0.537604,null,0.1953,null,null,null,null,null,null,null,null,null,null,0.4671,null,null,null,null,null,null,null,null,null,null,0.1897,null,null,null,…,null,null,null,0.5826,null,null,null,null,null,null,null,null,null,null,null,0.1292,null,null,null,null,null,null,null,null,1.22858,null,null,null,null,null,0.709,null,null,null,null,null,null
"""MIMAT0019869""",null,0.2117,null,null,0.1951,0.6706,null,null,null,0.535972,0.1953,null,0.1505,0.3557,null,null,null,null,null,null,null,null,0.2774,null,0.1523,null,null,null,null,null,null,null,null,null,null,null,…,null,1.056539,null,null,null,null,0.813429,null,null,0.61402,null,null,null,null,0.2227,null,null,null,0.5052,null,0.4137,null,null,0.774848,null,null,null,null,null,0.348594,null,null,null,0.7569,0.1298,null,null
"""MIMAT0019860""",null,null,null,null,null,null,0.403,0.4831,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.5006,null,null,0.7413,null,null,null,null,null,null,null,null,null,null,null,null
"""MIMAT0019862""",null,null,null,null,0.1951,null,null,null,null,null,0.3672,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.5772,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,0.3798,null,null,0.5018,null,0.2233,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.5619,null,null,0.4285,null,null,0.3034,0.5908,0.7569,null,null,0.2444
"""MIMAT0019863""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [36]:
mirna = mirna.drop_nulls()

In [37]:
mirna

sample,TCGA-OL-A66H-01,TCGA-3C-AALK-01,TCGA-AR-A1AH-01,TCGA-AC-A5EH-01,TCGA-EW-A2FW-01,TCGA-E9-A1R0-01,TCGA-BH-A0BL-01,TCGA-AR-A1AJ-01,TCGA-A7-A13G-01,TCGA-AC-A62Y-01,TCGA-E9-A1QZ-01,TCGA-E9-A1R2-01,TCGA-GM-A2DK-01,TCGA-AR-A255-01,TCGA-E2-A15T-01,TCGA-AO-A1KQ-01,TCGA-E9-A1N4-11,TCGA-C8-A1HL-01,TCGA-D8-A1JD-01,TCGA-D8-A1XA-01,TCGA-E9-A5FL-01,TCGA-BH-A1F2-11,TCGA-A7-A3RF-01,TCGA-A7-A3IY-01,TCGA-GM-A2D9-01,TCGA-D8-A3Z5-01,TCGA-E9-A1ND-01,TCGA-EW-A1P5-01,TCGA-E9-A1RD-01,TCGA-AR-A250-01,TCGA-E2-A153-11,TCGA-AR-A1AQ-01,TCGA-LD-A66U-01,TCGA-BH-A1EN-01,TCGA-AO-A1KO-01,TCGA-BH-A0AU-11,…,TCGA-AC-A2FG-01,TCGA-AR-A1AX-01,TCGA-AR-A1AL-01,TCGA-BH-A1ET-01,TCGA-C8-A12W-01,TCGA-D8-A13Z-01,TCGA-BH-A0BZ-11,TCGA-D8-A27K-01,TCGA-BH-A28Q-01,TCGA-E9-A1NG-11,TCGA-EW-A1J1-01,TCGA-BH-A18K-11,TCGA-C8-A1HG-01,TCGA-D8-A1XK-01,TCGA-A7-A4SA-01,TCGA-3C-AAAU-01,TCGA-A2-A4S0-01,TCGA-BH-A0DV-11,TCGA-D8-A27E-01,TCGA-AC-A8OQ-01,TCGA-BH-A0BS-11,TCGA-EW-A1PB-01,TCGA-D8-A1JU-01,TCGA-E9-A1R7-11,TCGA-D8-A147-01,TCGA-BH-A0BT-01,TCGA-E9-A1N8-01,TCGA-E2-A574-01,TCGA-E2-A1IG-01,TCGA-5L-AAT0-01,TCGA-E9-A1NH-01,TCGA-EW-A6SA-01,TCGA-AR-A24Q-01,TCGA-BH-A1F0-01,TCGA-E9-A1RB-11,TCGA-GM-A4E0-01,TCGA-AC-A5XU-01
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""MIMAT0000764""",5.827901,4.864784,5.897214,4.595051,6.562564,4.571719,5.333207,6.005498,5.630576,6.48816,6.82444,6.081916,4.258225,5.336978,4.747991,6.24437,5.958802,6.167489,4.586922,6.355424,5.574826,5.69638,4.846556,6.291839,3.982246,6.025131,6.35182,6.475671,2.83556,6.024522,4.619334,6.581411,3.563374,5.423291,5.142609,5.725651,…,4.399899,4.01072,5.187046,5.162911,4.274387,4.700227,6.613967,4.830681,3.959684,5.99095,6.145743,4.926559,6.914545,5.88053,5.230897,5.367761,4.902127,5.054248,4.537667,5.972747,5.56954,5.147002,4.945741,6.538525,4.995237,6.130271,5.127644,6.771455,4.917748,4.039321,5.996359,4.401192,5.540502,6.502696,6.483298,5.618528,6.027982
"""MIMAT0000761""",4.187861,3.515156,4.490978,3.43687,4.350379,2.185958,5.333233,5.247634,5.372102,4.367599,4.372397,4.743929,2.598367,4.834782,3.983686,5.591275,4.116495,4.911784,4.279915,4.826548,4.874407,2.978623,5.07236,5.322668,2.961671,4.423552,4.931451,5.369176,2.949095,4.595527,3.913039,6.53964,2.727175,5.278833,5.169571,3.63343,…,3.653753,2.794876,3.464465,4.350671,5.071458,5.57533,4.985277,2.954515,2.47914,4.957072,4.148866,3.973265,6.269288,6.237631,3.756858,5.062675,4.79262,3.633843,4.063994,3.958044,4.137778,4.908779,3.72229,4.868096,4.355925,6.49258,4.760409,6.269343,4.034822,2.944125,4.789642,4.070355,5.771872,5.561336,6.903009,3.314271,3.927624
"""MIMAT0000760""",3.641958,3.411793,5.214793,3.876836,5.293458,3.260027,5.054186,5.404873,6.671373,4.039516,4.382526,6.150091,4.809398,5.99779,5.664894,6.565282,4.419551,5.583124,5.494727,6.578404,3.88255,3.291912,6.393994,6.929057,3.258683,3.319367,5.834909,6.533904,3.128584,4.956835,3.276005,5.93404,2.218514,5.864073,4.984545,3.984876,…,3.865143,3.560671,4.335635,5.532129,5.10036,5.921946,4.635675,3.83577,3.24152,5.060076,6.297687,4.894549,6.217509,5.213432,3.347698,5.393741,4.618166,3.553559,4.903768,3.310791,4.292482,5.723988,4.812866,5.002175,5.972442,5.896529,6.058284,3.215237,5.637134,3.304883,6.106735,4.702955,4.771554,5.776288,5.100828,3.737816,3.780305
"""MIMAT0000763""",9.971574,8.13257,9.097876,7.166947,7.608273,7.227059,8.384586,8.904186,7.385219,8.609964,10.365238,11.167992,6.170031,7.002301,10.71348,7.529497,7.094889,7.810109,7.635586,7.442614,9.038773,8.511757,10.844651,7.694563,7.327571,7.796434,8.492257,8.258471,7.938741,7.968765,7.963503,7.916229,6.795001,7.842177,7.065751,6.347705,…,5.758397,7.586112,7.083565,6.819434,7.370583,7.615856,7.19032,6.26386,5.50811,7.624116,7.05965,7.720396,6.659074,8.571549,7.811599,11.100968,11.39911,7.209753,10.423676,

In [18]:
# load mirna db
mirna_db = pl.read_csv("mirna_db/miRNAs_miRandola_v02_2017.txt", separator="\t")

In [19]:
mirna_db

mature_mirna_from_literature,miRBase_Last_Version,miRBase_accession,miRBase_family,organism
str,str,str,str,str
"""bta-let-7""","""bta-let-7a-5p""","""MIMAT0003844""","""let-7""","""Bos taurus"""
"""bta-let-7-5p""","""bta-let-7a-5p""","""MIMAT0003844""","""let-7""","""Bos taurus"""
"""bta-let-7a""","""bta-let-7a-5p""","""MIMAT0003844""","""let-7""","""Bos taurus"""
"""bta-let-7b""","""bta-let-7b""","""MIMAT0004331""","""let-7""","""Bos taurus"""
"""bta-let-7c""","""bta-let-7c""","""MIMAT0004332""","""let-7""","""Bos taurus"""
…,…,…,…,…
"""rno-miR-322-3p…","""rno-miR-322-3p…","""MIMAT0000547""","""mir-322""","""male albino sp…"
"""rno-miR-324-3p…","""rno-miR-324-3p…","""MIMAT0000554""","""mir-324""","""male albino sp…"
"""rno-miR-421-3p…","""rno-miR-421-3p…","""MIMAT0017175""","""mir-95""","""male albino sp…"


In [30]:
mimat2mature = dict(mirna_db.select("miRBase_accession","mature_mirna_from_literature").iter_rows())
mimat2mature

{'MIMAT0003844': 'bta-let-7a',
 'MIMAT0004331': 'bta-let-7b',
 'MIMAT0004332': 'bta-let-7c',
 'MIMAT0003810': 'bta-let-7d',
 'MIMAT0004333': 'bta-let-7e',
 'MIMAT0003519': 'bta-let-7f',
 'MIMAT0003851': 'bta-let-7i',
 'MIMAT0009215': 'bta-miR-100',
 'MIMAT0003521': 'bta-miR-103',
 'MIMAT0003785': 'bta-miR-107',
 'MIMAT0003786': 'bta-miR-10a',
 'MIMAT0003538': 'bta-miR-125a',
 'MIMAT0003539': 'bta-mir-125b',
 'MIMAT0003540': 'bta-miR-126-3p',
 'MIMAT0009232': 'bta-miR-141',
 'MIMAT0009233': 'bta-miR-143',
 'MIMAT0003542': 'bta-miR-145',
 'MIMAT0024570': 'bta-miR-149-5p',
 'MIMAT0003524': 'bta-miR-151-3p',
 'MIMAT0003523': 'bta-miR-151-5p',
 'MIMAT0009238': 'bta-miR-152',
 'MIMAT0003815': 'bta-miR-17-5p',
 'MIMAT0003543': 'bta-miR-181a',
 'MIMAT0009244': 'bta-miR-182',
 'MIMAT0003819': 'bta-miR-191',
 'MIMAT0004335': 'bta-miR-195',
 'MIMAT0004337': 'bta-miR-19b',
 'MIMAT0003822': 'bta-miR-200a',
 'MIMAT0003842': 'bta-miR-200b',
 'MIMAT0003823': 'bta-miR-200c',
 'MIMAT0003545': 'bta-miR-2

In [38]:
mimatNames = mirna["sample"]

In [53]:
matureNames = np.array([None] * len(mimatNames), dtype=object)
matureNames

array([None, None, None, None, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, No

In [54]:
errors = 0

for i, mimat_name in enumerate(mimatNames):
    try:
        matureNames[i] = mimat2mature[mimat_name]
    except KeyError:
        errors += 1
        print(f"{mimat_name}", end=", ")

print(f"total errors {errors} / {len(mimatNames)}")

MIMAT0004485, MIMAT0004497, MIMAT0026477, MIMAT0019208, MIMAT0019731, MIMAT0016895, MIMAT0005878, MIMAT0015378, MIMAT0009451, MIMAT0004604, MIMAT0005797, MIMAT0019761, MIMAT0004563, MIMAT0004564, MIMAT0004799, MIMAT0004766, MIMAT0006790, MIMAT0000754, MIMAT0004927, MIMAT0005951, MIMAT0004599, MIMAT0004680, MIMAT0017985, MIMAT0004584, total errors 24 / 225


In [88]:
matureNames

array(['hsa-miR-339-5p', 'hsa-miR-324-5p', 'hsa-miR-331-3p',
       'hsa-miR-338-3p', 'hsa-miR-324-3p', 'hsa-miR-744*', 'hsa-miR-625*',
       'hsa-miR-628-5p', None, 'hsa-let-7a*', 'hsa-let-7b-3p',
       'hsa-miR-365b-3p', 'hsa-miR-598', 'hsa-miR-365a-3p',
       'hsa-miR-146a', 'hsa-miR-126*', 'hsa-miR-126-3p', 'hsa-miR-127-3p',
       'hsa-miR-134', 'hsa-miR-191-5p', 'hsa-miR-9-5p', 'hsa-miR-125a-5p',
       'hsa-miR-629', None, None, 'hsa-miR-21*', 'hsa-let-7c*', None,
       None, 'hsa-miR-361-5p', 'hsa-miR-362-5p', 'hsa-miR-193a-3p',
       'hsa-miR-186-5p', 'hsa-miR-185-5p', 'hsa-miR-150',
       'hsa-miR-149-5p', 'hsa-miR-1307-5p', 'hsa-let-7d-3p', None,
       'hsa-miR-486-5p', 'hsa-miR-484', None, 'hsa-miR-382',
       'hsa-miR-422b', 'hsa-miR-379-5p', 'hsa-miR-378*', 'hsa-miR-223',
       'hsa-miR-224-5p', 'hsa-miR-497', 'hsa-miR-181d', 'hsa-miR-339-3p',
       'hsa-miR-194-5p', 'hsa-miR-195-5p', 'hsa-miR-320a', None,
       'hsa-miR-542-3p', 'hsa-miR-451', None, 'hsa-miR-1

In [58]:
mirna = mirna.with_columns(mature=matureNames)

In [60]:
mirna = mirna.drop_nulls()

In [68]:
mirna_names_list = mirna["mature"].to_numpy().tolist()

In [77]:
from bipartite_gnn.preprocessing import ids_to_gene_names

In [85]:
ids_to_gene_names(
    ["ENSP00000000233", "ENSP00000356607"], "ensembl.protein"
)



['ARF5', 'RALGPS2']

In [90]:
ids_to_gene_names(
    ["ENSG00000010404", "ENSG00000010404", "ENSG00000089234"], "ensembl.gene"
)


[{'query': 'ENSG00000010404', '_id': '3423', '_score': 27.731798, 'alias': ['ID2S', 'MPS2', 'SIDS'], 'symbol': 'IDS'}, {'query': 'ENSG00000010404', '_id': '3423', '_score': 27.731901, 'alias': ['ID2S', 'MPS2', 'SIDS'], 'symbol': 'IDS'}, {'query': 'ENSG00000089234', '_id': '8315', '_score': 27.732014, 'alias': ['BRAP2', 'IMP', 'RNF52'], 'symbol': 'BRAP'}]


['IDS', 'IDS', 'BRAP']

In [104]:
ids_to_gene_names(
    ["MI0000764", "MI0000813"], "miRBase"
)

[{'query': 'MI0000764', '_id': '574031', '_score': 13.560646, 'alias': ['MIR-363', 'MIRN363', 'hsa-mir-363'], 'symbol': 'MIR363'}, {'query': 'MI0000813', '_id': '442898', '_score': 14.875502, 'alias': ['MIRN324', 'hsa-mir-324', 'mir-324'], 'symbol': 'MIR324'}]


['hsa-mir-363', 'hsa-mir-324']